# Notebook 1: Project Overview and Dataset

## Introduction

When users search for courses, they might ask questions like "What are the most recent courses on cognitive science?" or "What else does this instructor teach?" A standard search engine returns documents containing those keywords, but the user still has to connect the dots: filter courses by year manually, check instructor names across multiple results, and piece together relationships.

We have two ways to build a smarter discovery system.


**Approach A – Traditional RAG:** Retrieve relevant course descriptions using hybrid search, then feed that text to an LLM to generate an answer. It's simple and works well for straightforward questions.

**Approach B – Graph RAG:** Start with the same retrieved courses, but before sending them to the LLM, enrich them with structured context from a knowledge graph — topic hierarchies, instructor networks, co-occurrence relationships. Then let the LLM answer with full visibility into how courses, topics, and instructors connect.


Which approach produces better answers? This project implements both on MIT OpenCourseWare data and compares them side by side. Same questions, same retrieval entry point, same LLM. The only difference is the graph enrichment step.

The goal is practical: when building a real discovery system, where does a knowledge graph actually make a difference?

## Use Cases

Before designing the data model and ontology, it is important to define the use cases. In this project, three use cases were considered:

**1. Semantic course search**

Natural language queries to find relevant courses based on topic coverage and course description.

Example questions:

- "What courses cover machine vision?"
- "I want to learn about fairness and bias in machine learning"
- "What courses would help me understand how robots perceive the world?"


**2. Relationship-focused queries**

Questions that involve traversing relationships in the graph, instead of just matching text. This is where naive RAG struggles, but Graph RAG excels.

Example questions:
- "Which courses are also taught by this instructor?"
- "Which courses are related to this one?"
- "What are the most recent courses on a particular topic?"


**3. Graph analytics**

Quantitative analysis of the graph structure itself: identifying central topics, discovering course clusters, and computing similarity based on
graph topology.

Example questions:
- "Which topics are most central to the curriculum?"
- "What natural clusters of courses exist?"
- "Which courses are most similar based on shared topics?"

Use cases 1 and 2 are addressed in the RAG comparison (Notebook 4).
Use case 3 is addressed through graph algorithms (Notebook 5).

## Project Roadmap

| Notebook | Focus |
|---|---|
| **1. Project Overview and Dataset** | Goals, use cases, and data exploration |
| **2. Ontology Design** | From MIT's taxonomy to a graph ontology |
| **3. Building the Knowledge Graph** | Data loading into Neo4j and graph structure validation |
| **4. RAG Comparison** | Naive RAG vs Graph RAG compared using the same queries |
| **5. Graph Analytics** | Centrality, clustering, and similarity algorithms |

---
## The Data

For this project, I compiled a dataset consisting of 50 MIT OpenCourseWare (OCW) courses, parsed from MIT Learn webpages and converted into a structured CSV. Each course has a title, a text description, a list of topics from MIT's own taxonomy, and its instructors.

In [1]:
import pandas as pd
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path('..')

In [2]:
df = pd.read_csv(PROJECT_ROOT / 'data' / 'curated' / 'courses.csv')
print(f'{len(df)} courses, {len(df.columns)} fields')
df.head()

50 courses, 8 fields


,title,description,url,course_number,topics,instructors,availability,year
0,Advances in Computer Vision,This course dives into advanced concepts in co...,https://ocw.mit.edu/courses/6-8300-advances-in...,6-8300,"Data Science, Analytics & Computer Technology;...",Prof. Vincent Sitzmann,Anytime,2025
1,Affective Computing,This course instructs students on how to devel...,https://ocw.mit.edu/courses/mas-630-affective-...,MAS-630,"Data Science, Analytics & Computer Technology;...",Prof. Rosalind W. Picard,Anytime,2015
2,AI 101,Machine vision. Data wrangling. Reinforcement ...,https://ocw.mit.edu/courses/res-6-013-ai-101-f...,RES-6-013,"Data Science, Analytics & Computer Technology;...",Brandon Leshchinskiy,Anytime,2021
3,Ambient Intelligence,This course will provide an overview of a new ...,https://ocw.mit.edu/courses/mas-961-ambient-in...,MAS-961,"Data Science, Analytics & Computer Technology;...",Prof. Patricia Maes,Anytime,2005
4,Artificial Intelligence,This course introduces students to the basic k...,https://ocw.mit.edu/courses/6-034-artificial-i...,6-034,"Data Science, Analytics & Computer Technology;...",Patrick Henry Winston,Anytime,2010


### Field summary

| Field | Description |
|---|---|
| `title` | Course name |
| `description` | Free-text course description |
| `url` | Link to the OCW course page |
| `course_number` | MIT course number (e.g. 6-7960, RES-EC-001) |
| `topics` | Semicolon-separated list from MIT's topic taxonomy |
| `instructors` | Semicolon-separated list of instructors |
| `availability` | When the course is available |
| `year` | Year the course was taught |

### Exploring the data

In [3]:
# Topics and instructors per course
df['topic_count'] = df['topics'].str.split(';').apply(len)
df['instructor_count'] = df['instructors'].str.split(';').apply(len)

print(f'Topics per course:      min={df["topic_count"].min()}, '
      f'max={df["topic_count"].max()}, '
      f'mean={df["topic_count"].mean():.1f}')
print(f'Instructors per course: min={df["instructor_count"].min()}, '
      f'max={df["instructor_count"].max()}, '
      f'mean={df["instructor_count"].mean():.1f}')

Topics per course:      min=5, max=15, mean=7.7
Instructors per course: min=1, max=6, mean=1.7


In [4]:
# Most common topics
topic_freq = Counter()
for topics_str in df['topics']:
    for t in topics_str.split(';'):
        t = t.strip()
        if t:
            topic_freq[t] += 1

print(f'{len(topic_freq)} unique topics')
print('=====')
print('Top 10 Topics:')
print()
pd.DataFrame(topic_freq.most_common(10), columns=['topic', 'courses'])

44 unique topics
=====
Top 10 Topics:



,topic,courses
0,"Data Science, Analytics & Computer Technology",50
1,AI,50
2,Machine Learning,50
3,Computer Science,50
4,Engineering,50
5,Science & Math,13
6,Electrical Engineering,12
7,Cognitive Science,11
8,Algorithms and Data Structures,8
9,Humanities,7


The top 5 topics (Data Science Analytics & Computer Technology, AI, Machine Learning, Computer Science, and Engineering) appear in all 50 courses. They are useful for classification but uninformative for similarity or analytics in our current dataset, since they cannot distinguish one course from another. 

The remaining 39 topics are the interesting part of the structure: Science & Math (13), Electrical Engineering (12), Cognitive Science (11), and so on. We will return to their analysis when building the knowledge graph and evaluating retrieval.

In [5]:
# Instructors who teach multiple courses
inst_freq = Counter()
for inst_str in df['instructors']:
    for i in inst_str.split(';'):
        i = i.strip()
        if i:
            inst_freq[i] += 1

multi = {k: v for k, v in inst_freq.items() if v > 1}
print(f'{len(inst_freq)} unique instructors | {len(multi)} teach more than one course')
print()
pd.DataFrame(sorted(multi.items(), key=lambda x: -x[1]),
             columns=['instructor', 'courses'])

70 unique instructors | 14 teach more than one course



,instructor,courses
0,Prof. Leslie Kaelbling,4
1,Prof. Tomás Lozano-Pérez,3
2,Prof. Vincent Sitzmann,2
3,Patrick Henry Winston,2
4,Prof. Tomaso Poggio,2
5,Prof. Brian Charles Williams,2
6,Prof. Henry Lieberman,2
7,MIT RAISE,2
8,Prof. Bernhardt Trout,2
9,Prof. Isaac Chuang,2


In [6]:
# Clean up helper columns
df.drop(columns=['topic_count', 'instructor_count'], inplace=True)

---
## Next Steps

Now that we have an overview of the project goals and data, the next step is to design the ontology. In the next notebook, I use the MIT's topic taxonomy as a starting point and engineer it into a graph schema with typed nodes, hierarchical relationships, and co-occurrence edges.